# Evaluation : Perplexity

- Language Model [perplexity]
    - Character Level : 
        - Bigram [8.9391]
        - Trigram[4.5725]
        - Trigram + Add-k smoothing [4.4894]
    - Word Level : 
        - Bigram [61.1667]
        - Trigram[137.7248]
        - Trigram + Add-k smoothing [7.7377]

- Sparcity Problem 
    - Add-one (Laplace) smoothing : prob != 0
        - problem : word-level N-gram models -> vocab_siz 10,000 -> trigram's - 1 trillion possible entries == mostly 0

In [237]:
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer

model = GPT2LMHeadModel.from_pretrained('gpt2')
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')

text = "Large language models are fascinating."
inputs = tokenizer(text, return_tensors='pt')
loss = model(**inputs, labels=inputs["input_ids"]).loss
perplexity = torch.exp(loss)

print(f"Perplexity: {perplexity.item()}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Perplexity: 301.3665466308594


### Evaluation : `Perplexity` 
$$
PP(W) = P(w_1 w_2 \cdots w_N)^{-\frac{1}{N}}
= \frac{1}{\sqrt[N]{P(w_1 w_2 \cdots w_N)}}
$$

$$
PP(W)
=
\left(
\prod_{i=1}^{N}
P(w_i \mid w_1,\ldots,w_{i-1})
\right)^{-\frac{1}{N}}
$$


$$
PP(W)
=
\exp\!\left(
-\frac{1}{N}
\sum_{i=1}^{N}
\log P(w_i \mid w_1,\ldots,w_{i-1})
\right)
$$


- How surprised the model is
- Lower the better
- To judge if a model is 'learning' correctly
- Minimizing perplexity is the same as maximizing probability



# Character Level Trigram Language Model

In [256]:
import torch
import urllib.request
import re

# fetch and download "Berkeley Restaurant Project" Data
url = "https://raw.githubusercontent.com/wooters/berp-trans/master/transcript.txt"
response = urllib.request.urlopen(url)
data = response.read().decode('utf-8')

clean_lines = []
sentences = [line.split(" ", 1)[1] for line in data.strip().split('\n') if len(line.split(" ", 1)) > 1]
clean_text = " . ".join(sentences).lower()
clean_data = re.sub(r'[^a-z .]', '', clean_text)

In [257]:
# # Splitting Train / Test Data
split_idx = int(0.9 * len(clean_data))
d_train = clean_data[:split_idx]
d_test = clean_data[split_idx:]

In [258]:
# Preprocessing
chars = sorted(list(set(clean_data)))
# [' ', '.', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']

# Encoding
stoi = {s:i for i, s in enumerate(chars)}
itos = {i:s for i, s in enumerate(chars)}
vocab_size = len(stoi) # 28

#### Trigram character lebel LM + Add-K smoothing
- perplexity : 4.4894

In [259]:
k = 0.01
N = torch.full((vocab_size, vocab_size, vocab_size), k, dtype=torch.float32)

for i in range(len(d_train)-2):
	ch1 = d_train[i]
	ch2 = d_train[i+1]
	ch3 = d_train[i+2]
	
	ix1 = stoi[ch1]
	ix2 = stoi[ch2]
	ix3 = stoi[ch3]
	N[ix1][ix2][ix3] += 1
	
P = N.float()
P /= P.sum(dim=2, keepdim=True)

def generate(char1, char2, length=100):
	result = [char1, char2]

	for _ in range(length):
		ix1 = stoi[char1]
		ix2 = stoi[char2]

		temp = 0.5
		probs = P[ix1, ix2]
		probs = probs ** (1.0 / temp)
		probs = probs/probs.sum() # re-normalize

		next_idx = torch.multinomial(probs, num_samples=1, replacement=True)[0].item()

		char3 = itos[next_idx]
		# if char3 == ".":
		#     break

		result.append(char3)
		char1, char2 = char2, char3
		
	return "".join(result)

generate("i", " ", length=100)

'i want mack the th . to ther . ing to about plart on . in don food bre can restaurant . likell millike'

In [260]:
# Trigram character model perplexity : 4.4894
import numpy as np

log_likelihood = 0.0
trigram_count = 0

for i in range(len(d_test)-2):
	ch1, ch2, ch3 = d_test[i], d_test[i+1], d_test[i+2]
	ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]
	prob = P[ix1, ix2, ix3]
	if prob > 0:
		log_likelihood += torch.log(prob)
		trigram_count += 1
# print(log_likelihood) # tensor(-47290.9219)
# Calculate the perplexity
if trigram_count > 0:		# 31111
	avg_nll = -log_likelihood / trigram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid trigrams found in test set.")

Test set perplexity: 4.4894


#### Trigram LM
- perplexity : 4.5725

In [261]:
# Train with train data (d_train)
N = torch.ones((vocab_size, vocab_size, vocab_size), dtype=torch.float32) # (28,28,28)

for i in range(len(d_train)-2):
	ch1 = d_train[i]
	ch2 = d_train[i+1]
	ch3 = d_train[i+2]
	
	ix1 = stoi[ch1]
	ix2 = stoi[ch2]
	ix3 = stoi[ch3]
	N[ix1][ix2][ix3] += 1

# Normalize into probability Matrix P
P = N.float()
P /= P.sum(dim=2, keepdim=True)

In [262]:
def generate(char1, char2, length=100):
	result = [char1, char2]

	for _ in range(length):
		ix1 = stoi[char1]
		ix2 = stoi[char2]

		temp = 0.5
		probs = P[ix1, ix2]
		probs = probs ** (1.0 / temp)
		probs = probs/probs.sum() # re-normalize

		next_idx = torch.multinomial(probs, num_samples=1, replacement=True)[0].item()

		char3 = itos[next_idx]
		# if char3 == ".":
		#     break

		result.append(char3)
		char1, char2 = char2, char3
		
	return "".join(result)

In [263]:
generate("i", " ", length=100)

'i wout . i whave that thaver . i wan five the to eat me about the on dollaurant . i wout i would . i w'

In [264]:
# Trigram character model perplexity : 4.5725
import numpy as np

log_likelihood = 0.0
trigram_count = 0

for i in range(len(d_test)-2):
	ch1, ch2, ch3 = d_test[i], d_test[i+1], d_test[i+2]
	ix1, ix2, ix3 = stoi[ch1], stoi[ch2], stoi[ch3]
	prob = P[ix1, ix2, ix3]
	if prob > 0:
		log_likelihood += torch.log(prob)
		trigram_count += 1
# print(log_likelihood) # tensor(-47290.9219)
# Calculate the perplexity
if trigram_count > 0:		# 31111
	avg_nll = -log_likelihood / trigram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid trigrams found in test set.")

Test set perplexity: 4.5725


- Test set perplexity: 4.5725
- Perplexity is less then vocab_size (4.5725 < 28)

- Things to consider
1. Model complexity 
2. Smoothing Impacts
3. Data Sparsity
4. * Use the same test dataset to compare perplexity between models

#### Bigram LM
- perplexity : 8.9391

In [265]:
# Train Bigram LM 
N = torch.ones((vocab_size, vocab_size), dtype=torch.float32) # (28,28)

for i in range(len(d_train)-1):
	ch1 = d_train[i]
	ch2 = d_train[i+1]
	
	ix1 = stoi[ch1]
	ix2 = stoi[ch2]

	N[ix1][ix2] += 1

# Normalize into probability Matrix P
P = N.float()
P /= P.sum(dim=1, keepdim=True)

In [266]:
# perplexity of Bigram Character level LM
import numpy as np

log_likelihood = 0.0
bigram_count = 0

for i in range(len(d_test)-1):
	ch1, ch2 = d_test[i], d_test[i+1]
	ix1, ix2 = stoi[ch1], stoi[ch2]
	prob = P[ix1, ix2]
	if prob > 0:
		log_likelihood += torch.log(prob)
		bigram_count += 1

# Calculate the perplexity
if bigram_count > 0:		# 31112
	avg_nll = -log_likelihood / bigram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid bigrams found in test set.")

Test set perplexity: 8.9391


# Word Level LM 
- Bigram (Perplexity : 61.1667, vocab_size: 6612)
- Trigram (Perplexity : 137.7248, vocab_size: 6611)

- Why trigram perplexity is higher?
    - `Data Sparsity` : # of possible trigram comvinations grow cubically while trainig data remains constant. 
    - zero-probability transitions: probs 0~1 -> log negative -> prob~0 = very large (-) number
    -> penalty : very large (+) penalty
    - if test is is never seen in training - (Laplace smoothing) - tiny prob -> high penalty
    - `Overfitting` : memorizing sequence
    - `curse of dimensionality` = possible entries ($1603^3$)

- How to lower Trigram perplexity?
    - `Katz Backoff` - backoff strategy -> stabilize perplexity
    - `vocab pruning` (handling unknowns) : rare words -> <UNK>
    - `smoothing parameter tuning` = torch.ones(x) -> Good-Turing estimation or Kneser-Ney smoothing
    - `Data Augmentation or Cross-validation` to find the optiman N by splitting data into training, validation and test sets


In [267]:
words = clean_data.split()
unique_words = sorted(list(set(words)))

stoi = {s:i for i, s in enumerate(unique_words)}
itos = {i:s for i, s in enumerate(unique_words)}
vocab_size = len(unique_words) # 1603

In [268]:
N = torch.ones((vocab_size, vocab_size), dtype=torch.int32)

for i in range(len(words)-1):
	w1, w2 = words[i], words[i+1]
	ix1, ix2 = stoi[w1], stoi[w2]
	N[ix1, ix2] += 1
	
P = N.float()
P /= P.sum(dim=1, keepdim=True)

In [269]:
def generate_word(start_word, length=50):
	current_idx = stoi[start_word]
	result = [start_word]
	
	for _ in range(length):
		probs = P[current_idx]
		next_idx = torch.multinomial(probs, num_samples=1)[0].item()

		word = itos[next_idx]
		result.append(word)

		current_idx = next_idx

	return " ".join(result)

In [270]:
generate_word("i")

'i want that cancun gerties concerning southeast quality taqueriacancun polish fi tell omnivore friends asian off okay delicacy log bordeaux chi managed always mood new fine jeff weather instead poland telephone mermaid chowfun wannu sauls milano burmese hello bugs definitely hills thats soon christmas mediterranean ooh never having forty bar british'

### perflexity of Word level LM : 
- 61.1667 (vocab_size: 1603)

In [271]:
# convert teset string into a list of words
d_test_words = d_test.split()

In [272]:
import numpy as np

log_likelihood = 0.0
word_gram_count = 0

for i in range(len(d_test_words)-1):
	w1, w2 = d_test_words[i], d_test_words[i+1]

	# check if Both words exist in vocabn to prevent KeyError (new word in test data)
	if w1 in stoi and w2 in stoi:
		ix1, ix2 = stoi[w1], stoi[w2]
		prob = P[ix1, ix2]
		if prob > 0:
			log_likelihood += torch.log(prob)
			word_gram_count += 1
	else:
		continue #
# print(log_likelihood) tensor(-27199.1367)
# Calculate the perplexity
if word_gram_count > 0:		# 6612
	avg_nll = -log_likelihood / word_gram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid bigrams found in test set.")

Test set perplexity: 61.1667


#### Worl Level LM - Trigram

In [274]:
N = torch.ones((vocab_size, vocab_size, vocab_size), dtype=torch.int32)

for i in range(len(words)-2):
	w1, w2, w3 = words[i], words[i+1], words[i+2]
	ix1, ix2, ix3 = stoi[w1], stoi[w2], stoi[w3]
	N[ix1, ix2, ix3] += 1
	
P = N.float()
P /= P.sum(dim=2, keepdim=True)

In [275]:
def generate_word(start_word, length=50):
	current_idx = stoi[start_word]
	result = [start_word]
	
	for _ in range(length):
		probs = P[current_idx]
		next_idx = torch.multinomial(probs, num_samples=1)[0].item()

		word = itos[next_idx]
		result.append(word)

		current_idx = next_idx

	return " ".join(result)

generate_word("i")

'i lobster lock do milvia concerned entertainment both kings pesos roast weekday week northside wednesday hunan spats appreciate as never cancun tell preferred public accommodate touristy special name heike sit solano sunhongkong people ahead gilman since hes clam wed cut kinds vegi volga the dancing threw rasasayang goodbye same casbah sense'

In [276]:
import numpy as np

log_likelihood = 0.0
word_gram_count = 0

for i in range(len(d_test_words)-2):
	w1, w2, w3 = d_test_words[i], d_test_words[i+1], d_test_words[i+2]

	# check if Both words exist in vocabn to prevent KeyError (new word in test data)
	if w1 in stoi and w2 in stoi and w3 in stoi:
		ix1, ix2, ix3 = stoi[w1], stoi[w2], stoi[w3]
		prob = P[ix1, ix2, ix3]
		if prob > 0:
			log_likelihood += torch.log(prob)
			word_gram_count += 1
	else:
		continue #
print(word_gram_count)
# Calculate the perplexity
if word_gram_count > 0:		# 6612
	avg_nll = -log_likelihood / word_gram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid bigrams found in test set.")

6611
Test set perplexity: 137.7248


#### word Level LM - Trigram (Add-k)

In [279]:
k = 0.01
N = torch.full((vocab_size, vocab_size, vocab_size), k, dtype=torch.float32)

for i in range(len(words)-2):
	w1, w2, w3 = words[i], words[i+1], words[i+2]
	ix1, ix2, ix3 = stoi[w1], stoi[w2], stoi[w3]
	N[ix1, ix2, ix3] += 1
	
P = N.float()
P /= P.sum(dim=2, keepdim=True)

generate_word("i")

'i want thursday wonthaicuisine richpotsticker stores stop moes drop hows doing word christmas prompt true limousine thank chowder cheaper sunhongkong arinellpizza celebration should recording cause hongfu mexica benihama downstairs can seventy northside lavals northern grace wont personal basils threw health two smoking can waiters concord rockridge totally quickly gerties grammas thi'

In [281]:
import numpy as np

log_likelihood = 0.0
word_gram_count = 0

for i in range(len(d_test_words)-2):
	w1, w2, w3 = d_test_words[i], d_test_words[i+1], d_test_words[i+2]

	# check if Both words exist in vocabn to prevent KeyError (new word in test data)
	if w1 in stoi and w2 in stoi and w3 in stoi:
		ix1, ix2, ix3 = stoi[w1], stoi[w2], stoi[w3]
		prob = P[ix1, ix2, ix3]
		if prob > 0:
			log_likelihood += torch.log(prob)
			word_gram_count += 1
	else:
		continue #
print(word_gram_count)
# Calculate the perplexity
if word_gram_count > 0:		# 6612
	avg_nll = -log_likelihood / word_gram_count
	perplexity = torch.exp(avg_nll)
	print(f"Test set perplexity: {perplexity.item():.4f}")
else:
	print("No valid bigrams found in test set.")

6611
Test set perplexity: 7.7377


#### Interpolation (weighted average)
- combine the probability of the trigram, bigram and unigram.
- use next best guess when the specific trigram isn't available.

$$P_{\text{interp}}(w_3 | w_1, w_2) = \lambda_1 P(w_3 | w_1, w_2) + \lambda_2 P(w_3 | w_2) + \lambda_3 P(w_3)$$

#### Backoff (Fallback Approach)

- Logic:
1.  Check for the trigram $P(w_3 | w_1, w_2)$. If it exists, use it.
2.  If the count is 0, back off to the bigram $P(w_3 | w_2)$.
3.  If that is also 0, back off to the unigram $P(w_3)$.
- `Katz Backoff`: This is the most common professional implementation, where you use a "discounting" factor to ensure the total probability still sums to 1 after you switch levels.